# WX · 99 · wMNE versus LCMV under the frozen Xu protocol (saved outputs only)
Pairs every completed LCMV result with the matched wMNE result of the same mode, experiment, variant and setting
(same participants, folds, definitions; only the inverse differs) and lists the historical-graph results
separately. Descriptive: differences are reported, not ranked; the holdout has 13 participants.

**How to use:** *Runtime → Run all*. Nothing to edit. Allow Google Drive access when asked.

If Colab disconnects or the runtime ends, open the notebook again and *Run all* again: finished
work is kept on Google Drive and skipped, and the computation continues where it stopped.
Everything (checkpoints, results, figures, manifests) is written to Drive, never to the
temporary Colab disk.

**Suite:** matched wMNE counterpart of the D3 LCMV / Xu suite (methodology log entry 55, `docs/wmne_xu.md`). All
outputs are under `<outputs>/D3_WMNE_XU/`, with its own code pin `D3_WMNE_XU/CODE_COMMIT.txt`. The LCMV suite
(`D3_LCMV_XU/`: stores, results, pins), `graphsD3_v4_*` and D1/D2 are only read, never written. Safe to run while
LCMV experiments run in another Colab session (separate folders and pin; one GPU per session).

In [ ]:
# ===== D3 wMNE / Xu suite setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_WX_CODE = False   # leave False. True only if you were asked to switch this suite to newer code
# The wMNE / Xu suite has its own code pin, <outputs>/D3_WMNE_XU/CODE_COMMIT.txt (all WX notebooks run
# that commit); the LCMV suite pin D3_LCMV_XU/CODE_COMMIT.txt and all other pins are only reported.
BRANCH = 'claude/ecstatic-davinci-a2e8mh'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py', '--family', 'WX']
     + (['--update-code'] if UPDATE_WX_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh
from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
print('outputs on Drive:', OUT)
print('Dataset 3 folder:', PATHS['dataset3']['root'])
LX = OUT + '/D3_LCMV_XU'
WX = OUT + '/D3_WMNE_XU'
print('suite outputs:', WX, '| LCMV reference (read only):', LX)

In [ ]:
import glob
import pandas as pd
pd.set_option('display.max_rows', 400, 'display.width', 250)
sh("python scripts/wx_compare.py")
rep = sorted(glob.glob(WX + '/reports/compare_*'))[-1]
for f in ('paired.csv', 'historical.csv'):
    try:
        display(pd.read_csv(f'{rep}/{f}').round(3))
    except (FileNotFoundError, pd.errors.EmptyDataError):
        print(f, 'not available yet')